# lrl-toolkit — 0 → hero on Colab 🌍

Train a **genuine Welsh chat model** end-to-end — ingest → clean → tokenizer →
continued-pretrain → conversational data → fine-tune → evaluate → export — then
talk to it, all on a free Colab GPU.

This uses [`projects/welsh-colab.yaml`](https://github.com/jwilliamsresearch/lrl-toolkit/blob/main/projects/welsh-colab.yaml):

- **Base:** Qwen2.5-1.5B (QLoRA 4-bit) — already knows some Welsh.
- **Instruction data:** real — Dolly + OASST1 machine-translated to Welsh with **NLLB**. No Ollama, no API, no mock.
- **Time:** ~1–3 h on a free **T4**. The pipeline is **resumable** — if the session drops, re-run the run cell and it continues.

> ⚠️ **Set a GPU runtime first:** *Runtime → Change runtime type → T4 GPU* (or L4 / A100 on Colab Pro).

## 1 · Confirm a GPU is attached

In [ ]:
!nvidia-smi -L

## 2 · Clone the repo and install

Colab already ships PyTorch + CUDA, so installing the extras keeps that build and just adds the ML/data stack.

> If a shared dependency (e.g. `numpy`) gets upgraded and a later cell throws an import error, do *Runtime → Restart session* and continue from step 3 — the install persists.

In [ ]:
!git clone https://github.com/jwilliamsresearch/lrl-toolkit.git
%cd lrl-toolkit
!pip install -q -e ".[data,train,eval]"

## 3 · Sanity check

In [ ]:
!lrl version
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(no GPU — set a GPU runtime!)')

## 4 · (Optional) Persist results + tokens

Colab is ephemeral. Mount Drive to keep the trained model, and set `HF_TOKEN` only if you later add **gated** sources (OSCAR/FLORES/CulturaX) or want to `push_to_hub`. The demo config needs neither.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ['HF_TOKEN'] = 'hf_...'   # only for gated data / pushing to the Hub

## 5 · Look at the demo config

In [ ]:
!cat projects/welsh-colab.yaml

## 6 · Run the whole pipeline

One command runs all eight stages. It streams progress; the long parts are
`pretrain` (continued pretraining) and the NLLB translation inside `convdata`.

Resumable: re-running skips completed stages. To watch stage-by-stage instead,
use the commented cell below.

In [ ]:
!lrl run -c projects/welsh-colab.yaml

In [ ]:
# Stage-by-stage alternative (uncomment to run individually):
# !lrl ingest    -c projects/welsh-colab.yaml
# !lrl clean     -c projects/welsh-colab.yaml
# !lrl tokenizer -c projects/welsh-colab.yaml
# !lrl pretrain  -c projects/welsh-colab.yaml
# !lrl convdata  -c projects/welsh-colab.yaml
# !lrl finetune  -c projects/welsh-colab.yaml
# !lrl evaluate  -c projects/welsh-colab.yaml
# !lrl export    -c projects/welsh-colab.yaml

## 7 · Inspect the artifacts + report card

Everything lands under `projects/welsh-colab/artifacts/<stage>/`.

In [ ]:
import glob, json, os

print('=== artifact tree ===')
for root, _, files in os.walk('projects/welsh-colab/artifacts'):
    depth = root.count(os.sep) - 'projects/welsh-colab/artifacts'.count(os.sep)
    print('  ' * depth + os.path.basename(root) + '/')
    for f in sorted(files)[:8]:
        print('  ' * (depth + 1) + f)

print('\n=== evaluation report card ===')
for p in glob.glob('projects/welsh-colab/artifacts/evaluate/*.json'):
    print(p)
    print(json.dumps(json.load(open(p)), indent=2, ensure_ascii=False)[:2000])

## 8 · 🎉 Hero — chat with *your* Welsh model

The `export` stage merges the LoRA adapter into the base and writes it to
`.../export/merged`. Load it and talk to it. (First run downloads the base
weights if they aren't cached yet.)

In [ ]:
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

merged = Path('projects/welsh-colab/artifacts/export/merged')
assert merged.exists(), 'No merged model — make sure the export stage finished (step 6).'

tok = AutoTokenizer.from_pretrained(str(merged))
model = AutoModelForCausalLM.from_pretrained(
    str(merged), torch_dtype=torch.float16, device_map='auto')
model.eval()

def ask(prompt, max_new_tokens=200, temperature=0.7):
    msgs = [{'role': 'user', 'content': prompt}]
    if tok.chat_template:
        inputs = tok.apply_chat_template(
            msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
    else:
        inputs = tok(prompt, return_tensors='pt').input_ids.to(model.device)
    out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=temperature, top_p=0.9)
    return tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print(ask('Helo! Sut wyt ti heddiw?'))            # Hello! How are you today?
print('---')
print(ask('Beth yw prifddinas Cymru?'))            # What is the capital of Wales?
print('---')
print(ask('Ysgrifenna frawddeg fer am y tywydd.')) # Write a short sentence about the weather.

## 9 · Keep it / ship it

- **Save to Drive** (survives the session):

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r projects/welsh-colab/artifacts/export /content/drive/MyDrive/welsh-colab-export

- **Run it locally with Ollama:** `export/` also contains a GGUF quant and an
  Ollama `Modelfile` (`make_ollama_modelfile: true`). Download that folder, then
  `ollama create welsh -f Modelfile && ollama run welsh`.
- **Publish to the Hub:** set `export.push_to_hub: true` in the config and provide `HF_TOKEN`.

## Scaling up — from working to *good*

On **Colab Pro** (L4 24 GB / A100 40 GB), edit `projects/welsh-colab.yaml`:

- `base_model: qwen2.5-3b` or **`aya-expanse-8b`** (excellent for low-resource languages)
- `pretrain.max_steps: 3000+`, `pretrain.seq_len: 2048`, `ingest.max_gb: 5`
- `convdata.translate_limit: 5000`, and add higher-quality **native** data:

```yaml
convdata:
  translate: [dolly, oasst1]
  native_sets:
    - repo: CohereForAI/aya_dataset   # real, human-written target-language pairs
      instruction_field: inputs
      response_field: targets
      select_field: language
      select_value: Welsh             # check the dataset covers your language
      limit: 5000
```

**Any language** works the same way — `lrl languages` lists 33 built-ins, or add a YAML in `configs/languages/`. `lrl init <lang>` scaffolds a fresh project.

### Troubleshooting
- **CUDA OOM:** lower `pretrain.seq_len` / `finetune.max_seq_len` to 512, or use a smaller base.
- **Session disconnected:** just re-run step 6 — completed stages are skipped.
- **`lrl` not found:** use `!python -m lrl_toolkit.cli ...` instead.